In [19]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

In [5]:
data= fetch_california_housing()
df= pd.DataFrame(data.data, columns= data.feature_names)
df["Target"]= data.target

In [6]:
df.sample()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Target
2879,1.8819,41.0,4.412523,1.108656,1781.0,3.279926,35.37,-118.97,0.58


In [7]:
X= df.iloc[:, :-1]
y= df.iloc[:, -1]

In [8]:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state= 42)

In [12]:
lr= Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
dt= DecisionTreeRegressor(max_depth= 7, random_state= 42)
rf= RandomForestRegressor(n_estimators= 100, random_state=42)

model_estimators= [('lr', lr ), ('dt', dt), ('rf', rf)]

In [13]:
# In regression ensembles (like VotingRegressor), we simply: average the predicted 

vr= VotingRegressor(estimators= model_estimators)
vr.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lr', ...), ('dt', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


In [14]:
y_pred = vr.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

MSE: 0.3441408781688326


In [15]:
result= pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
print(result)

        Actual  Predicted
20046  0.47700   0.709636
3024   0.45800   1.123294
15663  5.00001   3.890062
20484  2.18600   2.554935
9814   2.78000   2.090550
...        ...        ...
15362  2.63300   2.057275
16623  2.66800   1.834599
18086  5.00001   4.724726
2144   0.72300   0.921953
3665   1.51500   1.774703

[4128 rows x 2 columns]


In [22]:
print("R2 Score:", np.round(r2_score(y_test, y_pred), 2))

R2 Score: 0.74


### Weighted Voting

In [24]:
wv = VotingRegressor(
    estimators= model_estimators,
    weights=[1, 1, 2]  # RF gets more importance
)
wv.fit(X_train, y_train)
y_pred_wv= wv.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred_wv))

MSE: 0.310372725122635


In [18]:
result= pd.DataFrame({"Actual": y_test, "Predicted": y_pred_wv})
print(result)

        Actual  Predicted
20046  0.47700   0.659602
3024   0.45800   1.027873
15663  5.00001   4.148361
20484  2.18600   2.548604
9814   2.78000   2.136335
...        ...        ...
15362  2.63300   2.109759
16623  2.66800   1.874362
18086  5.00001   4.733099
2144   0.72300   0.869987
3665   1.51500   1.743735

[4128 rows x 2 columns]


In [23]:
print("R2 Score:", np.round(r2_score(y_test, y_pred_wv), 2))

R2 Score: 0.76
